# Laboratorium 03 - ML-KEM w wersji demo

## Schemat ML-PKE

Schemat ML-PKE oparty jest na problemie M-LWE z następującymi parametrami:
- $q=3329=13\cdot 2^8+1$
- $n=256$
- $R=\mathbb{Z}[X]/(X^n+1),$ 
- $R_q=R/qR=\mathbb{Z}_q[X]/(X^n+1),$

Matematyczny model schematu jest następujący:

#### Generowanie kluczy KeyGen()$\rightarrow$(pk,sk)
- Macierz $A\in R^{k\times k}_q$ dana jest losowo z rozkładu jednostajnego
- Współczynniki sekretu $\pmb{s}\in R^k_q$ wybierane są z wycentrowanego rozkładu dwumianowego $B_{\eta_1}$
- Współczynniki wektora błędu $\pmb{e}\in R^k_q$ są wybierane również z $B_{\eta_1}$
- Klucz publiczny: $pk=\pmb{t}=A\pmb{s}+\pmb{e}$
- Klucz prywatny: $sk=\pmb{s}$



#### Szyfrowanie Enc(m,pk)$\rightarrow$c
- Chcemy zaszyfrować wielomian $m$ o współczynnikach z $\{0,1\}$
- Próbkujemy $\pmb{r}\in R^k_q$ z $B_{\eta_1}$
- Próbkujemy $\pmb{e}_1\in R^k_q$ z $B_{\eta_2}$
- Próbkujemy $e_2\in R$ z $B_{\eta_2}$
- Obliczamy $\pmb{u}=A^T\pmb{r}+\pmb{e}_1$
- Obliczamy $v=\pmb{t}^T\pmb{r}+e_2+\lceil(q/2)m\rfloor$
- $c=(\pmb{u},v)$

#### Deszyfrowanie Dec(c,sk)$\rightarrow$m
- Obliczamy $m_n=v-\pmb{s}^T\pmb{u}$
- Otrzymany wynik jest "zaszumiony", ponieważ obliczenia w rzeczywistości nie dają oryginalnej wiadomości $m$. Jednak dzięki zastosowanemu przeskalowaniu współczynniki $m_n$ są albo bliskie $\lfloor q/2\rceil$, (a zatem pierwotny współczynnik $m$ wynosił 1) albo są bliskie zera, co oznacza, że oryginalny współczynnik w $m$ wynosił 0.
- Wykonujemy odszumianie $m_n$ i otrzymujemy wiadomość $m$



## BabyKyber

Schemat BabyKyber to dokładnie realizacja matematycznego modelu schematu ML-PKE (czyli ML-PKE pozbawiony wszelkich dodatkowych funkcji, jak serializacje/deserializacje, funkcje hashujące, generowanie elementów pseudolosowych z ziaren losowości, obsługa NTT itp.). Oryginalnie powstał dla parametrów $q=17$, $n=4$, $k=2$, $\eta_1=\eta_2=1$.

### Dane testowe:

#### Generowanie klucza

$A=\left[\begin{matrix}
    6x^3+16x^2+16x+11&9x^3+4x^2+6x+3\\
    5x^3+3x^2+10x+1&6x^3+x^2+9x+15
\end{matrix}\right]$

$\mathbf{s}=(-x^3-x^2+x,-x^3-x)$

$\mathbf{e}=(x^2,x^2-x)$

$\mathbf{t}=A\mathbf{s}+\mathbf{e}:\ \ \mathbf{t}=(16x^3+15x^2+7,10x^3+12x^2+11x+6)$

#### Szyfrowanie

$m=1\cdot x^3+0\cdot x^2+1\cdot x+1=x^3+x+1$

$\mathbf{r}=(-x^3+x^2,x^3+x^2-1)$

$\mathbf{e_1}=(x^2+x,x^2)$

$e_2=-x^3-x^2$

$\mathbf{u}=A^T\mathbf{r}+\mathbf{e_1}:\ \ \mathbf{u}=(11x^3+11x^2+10x+3,4x^3+4x^2+13x+11)$

$v=\mathbf{t}^T\mathbf{r}+e_2+\lfloor\frac{q}{2}\rceil m:\ \ v=8x^3+6x^2+9x+16$

$\mathbf{c}=(\mathbf{u},v):\ \ \mathbf{c}=((11x^3+11x^2+10x+3,4x^3+4x^2+13x+11),8x^3+6x^2+9x+16)$

#### Deszyfrowanie
$m_n=v-\mathbf{s}^T\mathbf{u}:\ \ m_n=8x^3+14x^2+8x+6$


## Zadanie 1.

Zaimplementuj schemat BabyKyber z wykorzystaniem poniższej implementacji pierścienia $\mathbb{Z}_{q}[X]/(f)$. Przed wprowadzeniem losowości przetestuj implementację na powyższym zestawie danych testowych.

Oszacuj prawdopodobieństwo błędnego deszyfrowania. Zmień parametry tak, aby prawdopodobieństwo błędnego deszyfrowania spadło poniżej $10^{-5}$ (pamiętaj, że $q$ musi być liczbą pierwszą a $f$ wielomianem nierozkładalnym w $\mathbb{Z}_q$).

### Pierścień $\mathbb{Z}_{q}[X]/(f)$

In [1]:
import numpy as np

class Rq():
    q=17
    f=np.array([1,0,0,0,1]) #f=X^4+1

    def __init__(self,vec):
        _,r=np.polydiv(np.array(vec),self.f)
        self.vec=r%17

    def __repr__(self):
        return str(self.vec)

    def __add__(self,other):
        if isinstance(other,Rq):
            return Rq(np.polyadd(self.vec,other.vec))
        else:
            raise TypeError(f"unsupported operand type(s) for +: 'Rq' and '{type(other).__name__}'")

    def __sub__(self,other):
        if isinstance(other,Rq):
            return Rq(np.polyadd(self.vec,-1*other.vec))
        else:
            raise TypeError(f"unsupported operand type(s) for +: 'Rq' and '{type(other).__name__}'")


    def __mul__(self,other):
        if isinstance(other,Rq):
            return Rq(np.polymul(self.vec,other.vec))
        elif isinstance(other,int):
            return Rq(other*self.vec)
        else:
            raise TypeError(f"unsupported operand type(s) for *: 'Rq' and '{type(other).__name__}'")

    __rmul__=__mul__

In [ ]:
#TYPE YOUR CODE BELOW





## Schemat ML-KEM

Schemat ML-KEM jest schematem enkapsulacji klucza powstającym z ML-PKE za pomocą transformacji Fujisaki-Okamoto dla schematów IND-CPA z niezerowym prawdopodobieństwem błędnego deszyfrowania.

Pierwszą z transformacji jest transformacja T przekształcająca schemat PKE w schemat PKE$_1$.

![Transformacja T](T.png)

Następnie otrzymany schemat PKE$_1$ przekształcamy do schematu KEM transformacją $U$.

![Transformacja U](U.png)

Występujące w transformacjach funkcje G, H, J są funkcjami hashującymi:
- G=SHA3-512
- H=SHA3-256

a funkcja J to XOF J=SHAKE256( ,256).

## Zadanie 2.

Za pomocą powyższych transformacji i odpowiednio dobranych funkcji hashujących z biblioteki `hashlib` zaimplementuj schemat KEM oparty na BabyKyber. Empirycznie oszacuj prawdopodobieństwo, że ustalone zostaną różne klucze.

In [ ]:
#TYPE YOUR CODE BELOW


